# Linear SCM worlds with known ground truth

Every recovery test in axiom is written against a `sim` world. `LinearSCM` composes a
`CausalGraph` with edge coefficients, noise scales, latent scales for bidirected edges, and
intercepts; it simulates observational and interventional data and knows its own truth.

In [ ]:
from axiom.core import Spec
from axiom.sim import (
    LinearSCM, SCMError, coefficient_key, confounded_world, feedback_world, frontdoor_world,
    hidden_confounder_world, iv_world, latent_key, mediator_world, transport_pair,
)

In [ ]:
scm = LinearSCM.from_text("Z -> X: 0.8, Z -> Y: 1.5, X -> Y: 2.0, X <-> Y: 0.7", name="demo")
print(scm.graph)
print(scm.coefficients, scm.latent_sd)
print(coefficient_key("Z", "X"), latent_key("Y", "X"))
print("total effect X -> Y:", scm.total_effect("X", "Y"), "| direct:", scm.direct_effect("X", "Y"))

In [ ]:
frame = scm.simulate(5, seed=0)
print(frame.round(3))
do1 = scm.simulate(100_000, seed=0, intervene={"X": 1.0})
do0 = scm.simulate(100_000, seed=0, intervene={"X": 0.0})
print("E[Y|do(1)] - E[Y|do(0)] =", round(float(do1["Y"].mean() - do0["Y"].mean()), 3))
print("exact:", scm.interventional_mean("Y", intervene={"X": 1.0}) - scm.interventional_mean("Y", intervene={"X": 0.0}))

## Named worlds

Each documents its truth and what a naive estimator gets wrong.

In [ ]:
for w in (confounded_world, hidden_confounder_world, iv_world, frontdoor_world, mediator_world, feedback_world):
    s = w()
    print(f"{w.__name__:24s} {s.graph.to_text():40s} truth={s.total_effect('X', 'Y'):.2f} unmeasured={s.graph.unmeasured} feedback={s.graph.feedback}")
src, tgt = transport_pair()
print("transport pair differs in Z:", src.intercepts, "->", tgt.intercepts, "| selection:", tgt.graph.selection)

## Validation and identity

Coefficients must match the graph's edges exactly; the SCM is a `Spec`, so a world is a
hashable, serializable artifact a test can pin.

In [ ]:
try:
    LinearSCM.from_text("X -> Y: 1.0, X -> Y: 2.0")
except SCMError as e:
    print("refused:", e)
print(Spec.from_json(scm.to_json()) == scm, scm.content_hash()[:16])
print(scm.observed(frame).columns.tolist(), "| unmeasured dropped:", hidden_confounder_world().observed(hidden_confounder_world().simulate(3, seed=0)).columns.tolist())